# Adding Sinks to the Energy System Model

In the [previous notebook](../_01_initialize/_1_initialize_ESM.ipynb), we initialized an energy system model, which defines the basic structure of the energy system such as locations, commodities, and the temporal resolution.

In this notebook, we introduce **Source components**. Sources represent components that **produce a commodity and inject it into the energy system**. We focus on the most essential parameters required to define and understand a Source component, while more advanced and optional settings will be explained in subsequent notebooks.

Typical examples of sources include:

- renewable generation technologies such as wind turbines or photovoltaic systems
- fossil fuel extraction
- imports of commodities such as natural gas or electricity
- external supply such as hydrogen delivery

Sources therefore represent **entry points of commodities into the modeled system**.

The Sink component in FINE is closely related to the Source component and internally inherits from the same class. Therefore, the parameters described here are also relevant when modeling sinks.


## Load ESM

We first load the ESM from the [previous notebook](../_01_initialize/_1_initialize_ESM.ipynb).

In [3]:
import fine as fn
import fine.IOManagement.xarrayIO as xrIO
import pandas as pd
import numpy as np
from pathlib import Path
cwd = Path.cwd().resolve()
base_path = cwd.parents[2]
nc_file = base_path / "examples" / "Examples" / "NetCDF" / "esm_source.nc"

esM = xrIO.readNetCDFtoEnergySystemModel(nc_file)

## Add Sinks

### Electricity demand

We can now add an electricity demand as a first sink.

To do so we first generate a normalized daily load shape (`dailyProfile`) which defines typical relative demand levels for each hour of the day. This profile is repeated for 365 days, and small random variations (up to $+0.1$) are added to each hourly value to introduce variability. The resulting values are then scaled to approximate demand levels for two regions(`regionN` and `regionS`) using different multipliers (25 and 40, respectively). The final DataFrame is rounded to two decimal places and indexed by hourly timesteps.

In [4]:
dailyProfile = [
    0.6, # 12am - 1am
    0.6, # 1am - 2am
    0.6, # ...
    0.6,
    0.6,
    0.7,
    0.9,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    1,
    0.9,
    0.8 # 11pm - 12 am
    ]

electricityDemand = pd.DataFrame(
    [
        [(u + 0.1 * np.random.rand()) * 25, (u + 0.1 * np.random.rand()) * 40]
        for day in range(365)
        for u in dailyProfile
    ],
    index=range(8760), # Timesteps of the esM
    columns=["regionN", "regionS"],
).round(2)

esM.add(
    fn.Sink(
        esM = esM,
        name = "Electricity demand",
        commodity = "electricity",
        hasCapacityVariable = False,
        operationRateFix = electricityDemand
    )
)

In this example, the electricity demand is synthetically generated for demonstration purposes; in a typical energy system model, this data would instead come from real demand datasets or projections. These inputs must conform to one of the accepted formats as described [below](#operationratefix).

## Save the Energy System Model

In [5]:
cwd = Path.cwd().resolve()
base_path = cwd.parents[2]
nc_file = base_path / "examples" / "Examples" / "NetCDF" / "esm_source_sink.nc"

xrIO.writeEnergySystemModelToNetCDF(
    esM, outputFilePath=nc_file, overwriteExisting=True
)


Writing output to netCDF... 
Done. (0.1876 sec)


## General Structure of a Sink Instance

The Sink class inherits from the Source class; they share the same input parameters (see [Source class](_1_add_source.ipynb#required-arguments) for the parameter description) and differ only in the sign parameter, which is equal to $-1$ for Sink objects and $+1$ for Source objects.


### operationRateFix

`operationRateFix` indicates a fixed operation rate for each time step and location. It depends on `hasCapacityVariable` as follows:

- If ```hasCapacityVariable = True```, the values are given relative to the installed capacities (i.e. a value of 1 indicates a utilization of 100% of the capacity).
- If ```hasCapacityVariable = False```, the values are given as absolute values in form of the `commodityUnit` for each time step.

Type:

- None (default)
- Pandas DataFrame with positive ($\geq 0$) entries. The row indices have to match the in the energy system model specified time steps. The column indices have to equal the in the energy system model specified locations. The data in ineligible locations are set to zero.
- Dictionary with investment periods as keys and one of the two options above as values

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "NetCDF"))
from docstringTable import display_param_table


display_param_table(fn.Sink)